In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
from datetime import datetime
import math

# constants
str_project = '20231010-gen-xii'
str_task = 'ad_hoc'
str_subtask = 'gen_11_payload_parsing'
str_final_task = 'split_payloads'
int_n_requests_per_lambda = 100

# get today's date
str_date_today = datetime.today().strftime('%Y%m%d')

# import payloads
str_filename = 'df_payloads.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/days/{str_date_today}/payloads/{str_filename}'
df = pd.read_parquet(str_uri)

# keep only the most recent payload
df.drop_duplicates(
    subset=['ACCOUNTID'],
    keep='last', 
    inplace=True,
)

# sort
df.sort_values(by='REQUEST_DATETIME', ascending=True, inplace=True)

# get nrows
int_nrows = df.shape[0]

# divide by int_n_requests_per_lambda
int_n_lambdas = math.ceil(int_nrows / int_n_requests_per_lambda)

# create list to assign as new column
list_rows = list(np.tile(np.arange(1, int_n_lambdas+1), int_n_requests_per_lambda))

# make sure its the same length as df
list_rows = list_rows[:int_nrows]

# assign
df['rows'] = list_rows

# make df_idx.csv
list_rows = list(df['rows'].value_counts().index)
df_idx = pd.DataFrame({'row': list_rows})
df_idx.sort_values(by='row', ascending=True, inplace=True)

# save
str_filename = 'df_idx.csv'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename}'
df_idx.to_csv(str_uri, index=False)

# subset and save
for int_row in df_idx['row']:
    # subset
    df_tmp = df[df['rows'] == int_row].copy()
    # save
    str_filename = f'df_rows_{int_row}.gzip'
    str_uri = f's3://{str_project}/{str_task}/{str_subtask}/days/{str_date_today}/{str_final_task}/{str_filename}'
    df_tmp.to_parquet(str_uri, compression='gzip')

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxi-split-payloads

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  27.65kB
Step 1/7 : FROM python:3.9
 ---> ab7eeae5d25f
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> be6db65d342c
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 1da7040878d3
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> 8be0ddc50b91
Step 5/7 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> ff8050dbf920
Step 6/7 : COPY script.py .
 ---> 6c49df39a37f
Step 7/7 : CMD ["python3", "script.py"]
 ---> Running in 042a25b1e8e0
Removing intermediate container 042a25b1e8e0
 ---> d86a9f9173fb
Successfully built d86a9f9173fb
Successfully tagged genxi-split-payloads:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxi-split-payloads' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxi-split-payloads]
3af4f7f0b37b: Preparing
747b1d38a851: Preparing
2457140d41c6: Preparing
1fbb04935245: Preparing
126ef28403f3: Preparing
47e31a4d606a: Preparing
bf4966b4b813: Preparing
da15a2a37253: Preparing
89ca33c95b2e: Preparing
83db175c22e2: Preparing
c5d13b2949a2: Preparing
7e43f593c900: Preparing
47e31a4d606a: Waiting
bf4966b4b813: Waiting
072686bcd3db: Preparing
da15a2a37253: Waiting
89ca33c95b2e: Waiting
83db175c22e2: Waiting
c5d13b2949a2: Waiting
7e43f593c900: Waiting
072686bcd3db: Waiting
747b1d38a851: Layer already exists
1fbb04935245: Layer already exists
2457140d41c6: Layer already exists
126ef28403f3: Layer already exists
47e31a4d606a: Layer already exists
bf4966b4b813: Layer already exists
89ca33c95b2e: Layer already exists
da15a2a37253: Layer already exists
c5d13b2949a2: Layer already exists
83db175c22e2: Layer already exists
7e43f593c900: Layer already exists
072686bcd3db: Layer already e

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass